# 08 - Cleaning OECD Migration Database Indicators

This notebook filters and prepares OECD migration indicators related to Cameroon.

Analytical role:

- provides complementary evidence for Q1, Q2 and Q3
- keeps OECD indicators separate from UN DESA and Eurostat indicators
- standardizes selected measures for later integration into the analytical master tables

OECD indicators are used as supporting evidence because their definitions and coverage differ from the main sources used in each research question.


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

GLOBAL_PATH = DATA_RAW / "global"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
xls = pd.read_csv(GLOBAL_PATH / "oecd_migration_database_raw.csv")
xls.head()

In [ ]:
sheet1_raw = pd.read_csv(
    GLOBAL_PATH / "oecd_migration_database_raw.csv",
    header=None
)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw[
    (sheet1_raw.index == 0) | (sheet1_raw[0] == "CMR")
].copy()

sheet1_raw

In [ ]:
years = pd.to_numeric(sheet1_raw[9], errors="coerce")

sheet1_raw = sheet1_raw[
    (sheet1_raw.index == 0) |
    ((years >= 2015) & (years <= 2024))
].copy()


In [ ]:
sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw[
    (sheet1_raw.index == 0) | (sheet1_raw[5] == "Total")
].copy()

sheet1_raw = sheet1_raw.drop(columns=[11, 12])

sheet1_raw

In [ ]:
sheet1_raw.head(50)

In [ ]:
sheet1_raw = sheet1_raw.drop(columns=[0, 1, 2, 4, 5, 6, 8])

sheet1_raw = sheet1_raw.reset_index(drop=True)

sheet1_raw

In [ ]:
sheet1_raw.columns = sheet1_raw.iloc[0]

sheet1_raw = sheet1_raw.drop(index=0).reset_index(drop=True)

sheet1_raw.columns.name = None

sheet1_raw

In [ ]:
# Rename permit to permits
sheet1_raw = sheet1_raw.rename(
    columns={"Variable": "nature", "Value": "permits", "Country": "destination_country", "Year": "year"}
)

# Add source and nature columns
sheet1_raw["source"] = "oecd"

# Add the operational reference period
def classify_covid_period(year):
    if year < 2020:
        return "pre_covid"
    elif year in [2020, 2021]:
        return "covid"
    else:
        return "post_covid"

sheet1_raw["covid_period"] = sheet1_raw["year"].apply(classify_covid_period)

# Add a short variable without changing the original nature labels
nature_mapping = {
    "Inflows of foreign population by nationality": "inflow",
    "Outflows of foreign population by nationality": "outflow",
    "Inflows of asylum seekers by nationality": "asylum_inflow",
    "Stock of foreign-born population by country of birth": "stock_birth",
    "Stock of foreign population by nationality": "stock_nationality",
    "Acquisition of nationality by country of former nationality": "citizenship_acquisition",
    "Inflows of foreign workers by nationality": "worker_inflow",
    "Inflows of seasonal foreign workers by nationality": "seasonal_worker_inflow",
}

sheet1_raw["variable"] = sheet1_raw["nature"].map(nature_mapping)

missing_nature = sheet1_raw.loc[sheet1_raw["variable"].isna(), "nature"].drop_duplicates()
if not missing_nature.empty:
    raise ValueError(f"Natures non mappees: {missing_nature.tolist()}")

# Reorder columns
sheet1_raw = sheet1_raw[
    [
        "destination_country",
        "year",
        "permits",
        "source",
        "nature",
        "variable",
        "covid_period"
    ]
]

sheet1_raw

In [ ]:
sheet1_raw.to_csv(
    CLEANED_PATH / "oecd_migration_cleaned.csv",
    index=False
)

In [ ]:
nature_mapping = {
    "Inflows of foreign population by nationality": "inflow",
    "Outflows of foreign population by nationality": "outflow",
    "Inflows of asylum seekers by nationality": "asylum_inflow",
    "Stock of foreign-born population by country of birth": "stock_birth",
    "Stock of foreign population by nationality": "stock_nationality",
    "Acquisition of nationality by country of former nationality": "citizenship_acquisition",
    "Inflows of foreign workers by nationality": "worker_inflow",
    "Inflows of seasonal foreign workers by nationality": "seasonal_worker_inflow",
}

sheet1_raw["variable"] = sheet1_raw["nature"].map(nature_mapping)

sheet1_raw


In [ ]:
sheet1_raw = sheet1_raw.drop(columns=["nature"])

sheet1_raw = sheet1_raw.rename(
    columns={
        "variable": "nature",
        "permits": "value",
    }
)
sheet1_raw = sheet1_raw[
    [
        "destination_country",
        "year",
        "value",
        "source",
        "nature",
        "covid_period",
    ]
]

sheet1_raw